In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token)

api = HfApi(token=token)
info = api.model_info("meta-llama/Meta-Llama-3.1-8B")

print("Model access verified:", info.id)

Model access verified: meta-llama/Llama-3.1-8B


In [ ]:

import gc
import subprocess
import sys
import csv
import os
import torch

# Output directory
SAVE_DIR = "/kaggle/working/results/accuracy"
os.makedirs(SAVE_DIR, exist_ok=True)

# Belebele languages
languages = [
    "eng_Latn",
    "fra_Latn",
    "deu_Latn",
    "hin_Deva",
    "ita_Latn",
    "por_Latn",
    "spa_Latn",
    "tha_Thai",
]

MODEL = "Llama3.1"
DATASET = "belebele"
SUMMARY_FILE = os.path.join(SAVE_DIR, "run_summary.csv")

def run_experiment(model, dataset, language):
    command = [
        sys.executable,
        "src_patching/mcqa_evaluator.py",
        "--dataset", dataset,
        "--lang", language,
        "--llm_name", model,
        "--mc_key", "ABCD",
        "--batch_size", "1",
        "--save_dir", SAVE_DIR,
    ]

    try:
        print(f"\nRunning {dataset} | {language} | {model}")
        result = subprocess.run(command, check=False)
        print(f"Exit code: {result.returncode}")
        return result.returncode

    except Exception as e:
        print(f"Error running {language}: {e}")
        return -1

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("Notebook cleanup completed.")


# Load previous progress so successful languages are skipped
if os.path.exists(SUMMARY_FILE):
    with open(SUMMARY_FILE, newline="") as f:
        results = list(csv.DictReader(f))
else:
    results = []

completed = {
    row["language"]
    for row in results
    if row["dataset"] == DATASET
    and row["model"] == MODEL
    and row["status"] == "SUCCESS"
}

for i, language in enumerate(languages, start=1):
    if language in completed:
        print(f"Skipping {language}: already successful.")
        continue

    print(f"\n{'=' * 55}")
    print(f"Experiment {i}/{len(languages)}: {language}")
    print(f"{'=' * 55}")

    exit_code = run_experiment(MODEL, DATASET, language)
    status = "SUCCESS" if exit_code == 0 else "FAILED"

    # Replace any previous record for this experiment
    results = [
        row for row in results
        if not (
            row["dataset"] == DATASET
            and row["model"] == MODEL
            and row["language"] == language
        )
    ]

    results.append({
        "dataset": DATASET,
        "model": MODEL,
        "language": language,
        "status": status,
        "exit_code": exit_code,
    })

    # Save progress after each language
    with open(SUMMARY_FILE, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "dataset", "model", "language",
                "status", "exit_code"
            ],
        )
        writer.writeheader()
        writer.writerows(results)

    print(f"{language}: {status}")

print("\nAll experiments finished.")
print(f"Summary CSV: {SUMMARY_FILE}")
print(f"Detailed results: {SAVE_DIR}/Llama3.1/belebele/")


Skipping eng_Latn: already successful.
Skipping fra_Latn: already successful.

Experiment 3/8: deu_Latn

Running belebele | deu_Latn | Llama3.1


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Initializing evaluator for belebele...
Loading tokenizer from meta-llama/Meta-Llama-3.1-8B...
Tokenizer loaded
Loading model from meta-llama/Meta-Llama-3.1-8B...


Loading weights: 100%|██████████| 291/291 [00:09<00:00, 30.11it/s]


Model loaded

Starting evaluation...
Loading belebele dataset for language: deu_Latn
Dataset loaded
Building prompts...
Built 900 prompts
Example prompt:
Based on the below passage and the question, choose the best option. 

P: Stellen Sie sicher, dass Ihre Hand so entspannt wie möglich ist und Sie dennoch alle Töne korrekt treffen können. Versuchen Sie auch, mit den Fingern keine unnötig großen Bewegungen zu machen. So ermüden Sie sich möglichst wenig. Denken Sie daran, dass es nicht nötig ist, die Tasten mit viel Kraft anzuschlagen, um zusätzliche Lautstärke zu erzielen, wie es beim Klavier der Fall ist. Um die Lautstärke auf dem Akkordeon zu erhöhen, benutzt man den Blasebalg mit mehr Druck oder Geschwindigkeit.
Q: Was wäre dem Abschnitt zufolge kein guter Tipp, um gut Akkordeon zu spielen?
A: Für eine höhere Lautstärke wenden Sie mehr Kraft zum Anschlagen der Tasten auf
B: Beschränken Sie unnötige Bewegungen auf ein Minimum, um mehr Ausdauer zu haben
C: Achten Sie darauf, die Noten

In [ ]:
import os
import json

result_dir = (
    "/kaggle/working/results/accuracy/"
    "Llama3.1/belebele"
)

for lang in [
    "eng_Latn", "fra_Latn", "deu_Latn", "hin_Deva",
    "ita_Latn", "por_Latn", "spa_Latn", "tha_Thai"
]:
    filename = (
        f"belebele_eval_Llama3.1_{lang}_ABCD.jsonl"
    )
    path = os.path.join(result_dir, filename)

    if not os.path.exists(path):
        print(f"{lang}: MISSING")
        continue

    with open(path, "r") as f:
        records = [json.loads(line) for line in f if line.strip()]

    if records:
        accuracy = sum(r["acc"] for r in records) / len(records)
        print(
            f"{lang}: {len(records)}/900 examples, "
            f"accuracy={accuracy:.4%}"
        )
    else:
        print(f"{lang}: EMPTY FILE")